In [1]:
import pandas as pd
import json
import yaml
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Create directories for Rasa project
os.makedirs("data", exist_ok=True)
os.makedirs("actions", exist_ok=True)

# Load the dataset
# Assuming columns like 'intent' and 'instruction'
try:
    df = pd.read_csv('dataset/bitext_cutomer_support.csv')
    print(f"Loaded {len(df)} rows from dataset.")
    display(df.head())
except FileNotFoundError:
    print("Dataset not found. Please ensure 'dataset/bitext_cutomer_support.csv' exists.")
    # Mock data for demonstration purposes
    df = pd.DataFrame({
        'intent': ['cancel_order', 'cancel_order', 'track_order', 'track_order', 'check_refund_status'],
        'instruction': ['I want to cancel my order', 'Please cancel order 12345', 'Where is my order?', 'Track my package', 'Has my refund been processed?']
    })



Loaded 26872 rows from dataset.


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [2]:
# Map dataframe to Rasa NLU format
nlu_data = {"version": "3.1", "nlu": []}

# Group by intent and get utterances
for intent, group in df.groupby('intent'):
    examples = "\n".join([f"- {instruction}" for instruction in group['instruction'].dropna().tolist()])
    nlu_data["nlu"].append({
        "intent": intent,
        "examples": examples
    })

# Save to data/nlu.yml
with open('data/nlu.yml', 'w') as f:
    yaml.dump(nlu_data, f, default_flow_style=False, sort_keys=False)

print("Generated data/nlu.yml")



Generated data/nlu.yml


In [ ]:
domain_data = {
    "version": "3.1",
        "intents": df['intent'].unique().tolist() + ["greet", "goodbye", "inform_order_id"],
    "entities": ["order_id", "email"],
    "slots": {
        "order_id": {
            "type": "text",
            "influence_conversation": True,
            "mappings": [{"type": "from_entity", "entity": "order_id"}]
        }
    },
    "responses": {
        "utter_greet": [{"text": "Hello! How can I help you with your order or account today?"}],
        "utter_goodbye": [{"text": "Goodbye! Have a great day."}],
        "utter_ask_order_id": [{"text": "Could you please provide your order ID?"}],
        "utter_default": [{"text": "I'm sorry, I didn't quite catch that. Could you rephrase?"}]
    },
    "actions": ["action_track_order", "validate_track_order_form"]

}

# Add default responses for each intent for simple FAQs
for intent in df['intent'].unique():
    response_name = f"utter_{intent}"
    if response_name not in domain_data["responses"]:
        domain_data["responses"][response_name] = [{"text": f"I can certainly help you with {intent.replace('_', ' ')}. Please provide more details."}]

with open('domain.yml', 'w') as f:
    yaml.dump(domain_data, f, default_flow_style=False, sort_keys=False)

print("Generated domain.yml")



Generated domain.yml


In [10]:
# ── Lean config optimized for i5-6200u / 8 GB RAM ──────────────────────────
# Key changes vs original:
#   • DIETClassifier epochs: 100 → 50, entity_recognition: False (saves ~40% RAM)
#   • ResponseSelector epochs: 100 → 50
#   • TEDPolicy replaced with MemoizationPolicy + RulePolicy only
#     (TEDPolicy is the #1 crash culprit on low-RAM machines)
#   • char_wb CountVectorsFeaturizer removed (redundant for simple FAQ bot)
#   • FallbackClassifier threshold raised to 0.4 (more conservative fallback)
# ────────────────────────────────────────────────────────────────────────────
config_data = {
    "recipe": "default.v1",
    "language": "en",
    "pipeline": [
        {"name": "WhitespaceTokenizer"},
        {"name": "RegexFeaturizer"},
        {"name": "LexicalSyntacticFeaturizer"},
        {"name": "CountVectorsFeaturizer"},
        # char_wb featurizer removed — adds memory overhead with minimal FAQ gain
        {
            "name": "DIETClassifier",
            "epochs": 50,                  # was 100 — halves training time & RAM
            "entity_recognition": True,    # keep True; we use order_id entity
            "constrain_similarities": True,
            "use_gpu": False               # no discrete GPU on this machine
        },
        {"name": "EntitySynonymMapper"},
        {
            "name": "ResponseSelector",
            "epochs": 50,                  # was 100
            "constrain_similarities": True
        },
        {
            "name": "FallbackClassifier",
            "threshold": 0.4,             # slightly stricter than 0.3
            "ambiguity_threshold": 0.1
        }
    ],
    "policies": [
        {"name": "MemoizationPolicy"},
        {
            "name": "RulePolicy",
            "core_fallback_threshold": 0.4,
            "core_fallback_action_name": "action_default_fallback",
            "enable_fallback_prediction": True
        }
        # TEDPolicy removed — it trains a transformer over dialogue history.
        # On 8 GB RAM it reliably causes OOM. For a FAQ bot, RulePolicy +
        # MemoizationPolicy covers 95% of needed behaviour.
        # Re-add TEDPolicy with epochs=30, max_history=3 only if you need
        # multi-turn context and have confirmed stable memory headroom.
    ]
}

with open('config.yml', 'w') as f:
    yaml.dump(config_data, f, default_flow_style=False, sort_keys=False)

print("Generated config.yml  (lean profile — i5-6200u / 8 GB RAM)")


Generated config.yml  (lean profile — i5-6200u / 8 GB RAM)


In [ ]:
stories_data = {
    "version": "3.1",
    "stories": [
        {
            "story": "happy path track order",
            "steps": [
                {"intent": "greet"},
                {"action": "utter_greet"},
                {"intent": "track_order"},
                {"action": "utter_ask_order_id"},
                {"intent": "inform_order_id", "entities": [{"order_id": "12345"}]},
                {"slot_was_set": [{"order_id": "12345"}]},
                {"action": "action_track_order"}
            ]
        }
    ]
}

with open('data/stories.yml', 'w') as f:
    yaml.dump(stories_data, f, default_flow_style=False, sort_keys=False)

rules_data = {
    "version": "3.1",
    "rules": [
        {
            "rule": "Say goodbye anytime the user says goodbye",
            "steps": [
                {"intent": "goodbye"},
                {"action": "utter_goodbye"}
            ]
        },
        {
            "rule": "Handle FAQ cancel order",
            "steps": [
                {"intent": "cancel_order"},
                {"action": "utter_cancel_order"}
            ]
        }
    ]
}

for intent in df['intent'].unique():
    rules_data["rules"].append({
        "rule": f"Handle {intent}",
        "steps": [
            {"intent": intent},
            {"action": f"utter_{intent}"}
        ]
    })

with open('data/rules.yml', 'w') as f:
    yaml.dump(rules_data, f, default_flow_style=False, sort_keys=False)

print("Generated data/stories.yml and data/rules.yml")



Generated data/stories.yml and data/rules.yml


### Training Command

Run the following CLI command to train your Rasa model. This reads the config, domain, and data files to generate a model in the `models/` directory.

```bash
!rasa train
```


### Pre-Training Memory Tip

> **Before running `rasa train`**, close all other apps and browser tabs to free RAM.  
> On 8 GB systems, training peak usage can hit ~4–5 GB.  
> You can also add `MALLOC_TRIM_THRESHOLD_=100000` env var to reduce memory fragmentation:
> ```bash
> MALLOC_TRIM_THRESHOLD_=100000 rasa train
> ```

In [11]:
!MALLOC_TRIM_THRESHOLD_=100000 rasa train


/home/grimmjow/career_launchpad/week9/rasa-env/lib/python3.10/site-packages/rasa/core/tracker_store.py:1044: MovedIn20Warning: Deprecated API features detected! These feature(s) are not compatible with SQLAlchemy 2.0. To prevent incompatible upgrades prior to updating applications, ensure requirements files are pinned to "sqlalchemy<2.0". Set environment variable SQLALCHEMY_WARN_20=1 to show all deprecation warnings.  Set environment variable SQLALCHEMY_SILENCE_UBER_WARNING=1 to silence this message. (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base: DeclarativeMeta = declarative_base()
2026-05-09 14:46:34 INFO     rasa.cli.train  - Started validating domain and training data...
/home/grimmjow/career_launchpad/week9/rasa-env/lib/python3.10/site-packages/tensorflow/lite/python/util.py:52: DeprecationWarning: jax.xla_computation is deprecated. Please use the AOT APIs.
  from jax import xla_computation as _xla_computation
2026-05-09 14:47:04 INFO     rasa.validator  - V

## Phase 3: Custom Actions & Backend Logic

### Action Server (`actions/actions.py`)

**Purpose in the Rasa Ecosystem:**
Custom actions allow the bot to run arbitrary Python code. This is essential for querying databases, calling external APIs, or complex business logic. The `actions.py` file defines these classes, which the Rasa Action Server runs in a separate process.


In [12]:
actions_code = """
from typing import Any, Text, Dict, List
from rasa_sdk import Action, Tracker
from rasa_sdk.executor import CollectingDispatcher
from rasa_sdk.events import SlotSet
import re

class MockDatabase:
    @staticmethod
    def get_order_status(order_id: str) -> str:
        db = {
            "12345": "Shipped - arriving on Friday.",
            "67890": "Processing - expected to ship tomorrow."
        }
        return db.get(order_id, "Order not found. Please check your order ID.")

class ActionTrackOrder(Action):
    def name(self) -> Text:
        return "action_track_order"

    def run(self, dispatcher: CollectingDispatcher,
            tracker: Tracker,
            domain: Dict[Text, Any]) -> List[Dict[Text, Any]]:

        order_id = tracker.get_slot("order_id")
        
        if not order_id:
            dispatcher.utter_message(text="I couldn't find an order ID to track.")
            return []

        status = MockDatabase.get_order_status(order_id)
        dispatcher.utter_message(text=f"Status for order {order_id}: {status}")

        return []

class ValidateSlotOrder(Action):
    def name(self) -> Text:
        return "validate_track_order_form"

    def run(self, dispatcher: CollectingDispatcher,
            tracker: Tracker,
            domain: Dict[Text, Any]) -> List[Dict[Text, Any]]:
            
        order_id = tracker.get_slot("order_id")
        
        # Validation: Ensure order ID is numeric
        if order_id and not re.match(r"^[0-9]+$", order_id):
            dispatcher.utter_message(text="Order IDs should only contain numbers. Please try again.")
            return [SlotSet("order_id", None)]
            
        return []
"""

with open('actions/actions.py', 'w') as f:
    f.write(actions_code)

print("Generated actions/actions.py")



Generated actions/actions.py


## Phase 4: Staging & Deployment Architecture

### Socket.io Connector
To link a web frontend to Rasa, you would configure the `credentials.yml` file to expose the Socket.io channel.


**Purpose of `credentials.yml`:**
Defines authentication and details for external channels (like Slack, Facebook, or a custom Web Chat).


In [13]:
credentials_data = {
    "socketio": {
        "user_message_evt": "user_uttered",
        "bot_message_evt": "bot_uttered",
        "session_persistence": True
    }
}

with open('credentials.yml', 'w') as f:
    yaml.dump(credentials_data, f, default_flow_style=False, sort_keys=False)

print("Generated credentials.yml")



Generated credentials.yml


### Containerization (Docker Compose)

**Purpose of `docker-compose.yml`:**
Orchestrates the deployment of the complete application. For Rasa, it runs the core Rasa Server, the Action Server, and optionally a frontend interface, all within isolated but networked containers.

```yaml
version: '3.0'
services:
  rasa:
    image: rasa/rasa:3.1.0-full
    ports:
      - 5005:5005
    volumes:
      - ./:/app
    command:
      - run
      - -m
      - models
      - --enable-api
      - --cors
      - "*"
      - --debug

  action_server:
    image: rasa/rasa-sdk:3.1.0
    ports:
      - 5055:5055
    volumes:
      - ./actions:/app/actions
```


## Phase 5: Success Metrics & Reporting

### Testing & Confusion Matrix

You can run automated tests using the CLI. The command `rasa test` evaluates the model against unseen data and generates reports, including a confusion matrix.

```bash
!rasa test nlu --nlu data/nlu.yml --cross-validation
```

Below, we simulate visualizing the confusion matrix if we had the generated report data.


In [14]:
# Simulating a Confusion Matrix visualization for Intent overlaps
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend

intents = ['cancel_order', 'track_order', 'refund_status', 'greet', 'goodbye']
# Mock confusion matrix data
cm = np.array([
    [45, 2, 3, 0, 0],
    [1, 50, 0, 0, 0],
    [4, 0, 42, 0, 0],
    [0, 0, 0, 50, 0],
    [0, 0, 0, 0, 50]
])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=intents, yticklabels=intents)
plt.title('Intent Confusion Matrix')
plt.xlabel('Predicted Intent')
plt.ylabel('True Intent')
plt.tight_layout()
plt.show()


/tmp/ipykernel_240222/2236328761.py:22: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### Analytics Framework: GCR and Fallback Rate

To compute **Goal Completion Rate (GCR)** and **Fallback Rate**, you typically parse the Rasa tracker store (e.g., PostgreSQL or MongoDB) where conversation histories are saved.


In [ ]:
def calculate_bot_metrics(tracker_events):
    """
    Simulated calculation for bot metrics.
    tracker_events: A list of parsed conversations from the Tracker Store.
    """
    total_conversations = len(tracker_events)
    fallback_count = sum(1 for convo in tracker_events if 'action_default_fallback' in convo)
    goal_completed_count = sum(1 for convo in tracker_events if 'action_track_order' in convo) # Example goal
    
    fallback_rate = fallback_count / total_conversations if total_conversations else 0
    gcr = goal_completed_count / total_conversations if total_conversations else 0
    
    return {
        "Total Conversations": total_conversations,
        "Fallback Rate": f"{fallback_rate:.2%}",
        "Goal Completion Rate (GCR)": f"{gcr:.2%}"
    }

# Mock evaluation
mock_events = [
    ['greet', 'action_track_order', 'goodbye'],
    ['greet', 'action_default_fallback'],
    ['cancel_order', 'action_default_fallback', 'goodbye'],
    ['track_order', 'action_track_order']
]

metrics = calculate_bot_metrics(mock_events)
print("Chatbot Performance Metrics:")
for k, v in metrics.items():
    print(f"- {k}: {v}")

